In [2]:
import os
import shutil
import json
import pandas as pd
from glob import glob

# --- CONFIGURATION ---
BASE_DIR = "/scratch/pbergere/COBRE"
SOURCE_DIR = os.path.join(BASE_DIR, "COBRE_nifti")
BIDS_ROOT = os.path.join(BASE_DIR, "bids_dataset")
PHENO_FILE = os.path.join(BASE_DIR, "COBRE_phenotypic_data.csv")

# --- CALCUL DU SLICE TIMING (Siemens Interleaved IMPAIR) ---
# TR=2.0s, 33 slices.
# Règle Siemens Impair: 1, 3, 5... (t=0, dt...) puis 2, 4, 6... (t=mid, mid+dt...)
# Donc la Coupe 1 (Index 0) est acquise à 0.0s.

tr = 2.0
n_slices = 33  # CORRECTION: 33 coupes
dt = tr / n_slices
slice_timing = [0.0] * n_slices

# Passe 1: Coupes impaires (Indices 0, 2, 4...) -> Acquises en PREMIER
for i, slice_idx in enumerate(range(0, n_slices, 2)):
    slice_timing[slice_idx] = i * dt

# Passe 2: Coupes paires (Indices 1, 3, 5...) -> Acquises APRÈS
# Nombre de coupes dans la passe 1 = ceil(33/2) = 17
n_first_pass = (n_slices + 1) // 2 
for i, slice_idx in enumerate(range(1, n_slices, 2)):
    slice_timing[slice_idx] = (i + n_first_pass) * dt

# --- METADATA ---
T1W_METADATA = {
    "Modality": "MR",
    "MagneticFieldStrength": 3,
    "Manufacturer": "Siemens",
    "ManufacturersModelName": "TrioTim",
    "RepetitionTime": 2.53,
    "EchoTime": 0.00164,
    "FlipAngle": 7,
    "BodyPartExamined": "BRAIN",
    "ProcedureStepDescription": "mprage_5e"
}

FUNC_METADATA = {
    "Modality": "MR",
    "MagneticFieldStrength": 3,
    "Manufacturer": "Siemens",
    "ManufacturersModelName": "TrioTim",
    "RepetitionTime": 2.0,
    "EchoTime": 0.029,
    "FlipAngle": 75,
    "TaskName": "rest",
    "BodyPartExamined": "BRAIN",
    "ProcedureStepDescription": "RSTpre_V01_R01",
    "SliceTiming": slice_timing  # Utilise le timing corrigé
}

def setup_bids_dirs():
    if os.path.exists(BIDS_ROOT):
        print(f"Attention: Le dossier {BIDS_ROOT} existe déjà.")
    os.makedirs(BIDS_ROOT, exist_ok=True)
    
    desc = {
        "Name": "COBRE",
        "BIDSVersion": "1.8.0",
        "DatasetType": "raw",
        "Authors": ["The Mind Research Network"]
    }
    with open(os.path.join(BIDS_ROOT, "dataset_description.json"), 'w') as f:
        json.dump(desc, f, indent=4)

def process_participants():
    print("Traitement des participants...")
    if not os.path.exists(PHENO_FILE):
        return

    df = pd.read_csv(PHENO_FILE)
    df = df.rename(columns={
        df.columns[0]: 'participant_id',
        'Current Age': 'age',
        'Gender': 'sex',
        'Handedness': 'handedness',
        'Subject Type': 'group',
        'Diagnosis': 'diagnosis'
    })

    # Formatage propre
    df['participant_id'] = df['participant_id'].astype(str).apply(lambda x: f"sub-{x.zfill(7)}")
    df['sex'] = df['sex'].map({'Male': 'M', 'Female': 'F'})
    df['handedness'] = df['handedness'].map({'Right': 'R', 'Left': 'L', 'Both': 'A'})
    
    df.to_csv(os.path.join(BIDS_ROOT, "participants.tsv"), sep='\t', index=False)

def convert_subjects():
    print("Conversion des images...")
    subjects = sorted(glob(os.path.join(SOURCE_DIR, "004*")))
    
    for subj_path in subjects:
        subj_id_raw = os.path.basename(subj_path)
        subj_label = f"sub-{subj_id_raw}"
        
        anat_dir = os.path.join(BIDS_ROOT, subj_label, "anat")
        func_dir = os.path.join(BIDS_ROOT, subj_label, "func")
        os.makedirs(anat_dir, exist_ok=True)
        os.makedirs(func_dir, exist_ok=True)
        
        # Copie Anat
        anat_src = glob(os.path.join(subj_path, "session_1", "anat_*", "mprage.nii.gz"))
        if anat_src:
            shutil.copy2(anat_src[0], os.path.join(anat_dir, f"{subj_label}_T1w.nii.gz"))
            with open(os.path.join(anat_dir, f"{subj_label}_T1w.json"), 'w') as f:
                json.dump(T1W_METADATA, f, indent=4)

        # Copie Func
        func_src = glob(os.path.join(subj_path, "session_1", "rest_*", "rest.nii.gz"))
        if func_src:
            shutil.copy2(func_src[0], os.path.join(func_dir, f"{subj_label}_task-rest_bold.nii.gz"))
            with open(os.path.join(func_dir, f"{subj_label}_task-rest_bold.json"), 'w') as f:
                json.dump(FUNC_METADATA, f, indent=4)

if __name__ == "__main__":
    setup_bids_dirs()
    process_participants()
    convert_subjects()
    print("Conversion terminée.")

Attention: Le dossier /scratch/pbergere/COBRE/bids_dataset existe déjà.
Traitement des participants...
Conversion des images...
Conversion terminée.
